# Quantum-simulated and classical enhancement of simulated low-dose chest radiographs

This notebook provides the **source implementation** accompanying the accepted conference study on conventional, QPAR, and hybrid chest-X-ray enhancement. It retains the original method implementations and adds source-faithful data checks and optional metric exports. It is **not a record of a newly rerun benchmark**: all code-cell outputs have been cleared before publication.

**Clinical dataset required.** Download the Kaggle Chest X-Ray Pneumonia collection separately. Set `CHEST_XRAY_DIR` (or change `CHEST_DIR` in the loading cell) to its local folder with the JPEG images. The clinical experiment **must never** silently fall back to a synthetic phantom. Follow `../README.md` for setup.

**Quantum implementation.** QPAR is constructed with Qiskit, then evaluated with a precomputed dense unitary `Operator(circuit).data` and NumPy matrix-vector multiplication. The timing therefore measures **dense-unitary state-vector emulation**, not an Aer simulator run or physical quantum hardware.

**Limitations.** The paper's clinical comparison does not include learned denoising baselines or direct diagnostic evaluation. Per-image clinical outputs were not contained in the original uploaded notebook; rerunning the clinical benchmark can generate them. A previously drawn mixing-strength figure was omitted from the manuscript due to missing source provenance, and is not included here.


## 1. Setup

**Install dependencies in the notebook's Python environment before running** (see project root `README.md`):

```bash
python -m pip install -r requirements.txt
```

Restart the kernel after installation. Then run the cells sequentially. The full benchmark requires the separate clinical image dataset; no patient images are in this repository.

In [ ]:
import os
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.measure import shannon_entropy
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator

rng_global = np.random.default_rng(42)
print("qiskit", __import__("qiskit").__version__, "| opencv", cv2.__version__)

## 2. Why the usual $R_y$/CNOT ansatz cannot smooth

Amplitude encoding maps a patch of $N_p$ pixels onto $n=\log_2 N_p$ qubits:

$$|\psi\rangle = \sum_{r} c_r |r\rangle, \qquad \sum_r |c_r|^2 = 1$$

Basis state $|r\rangle$ indexes a **pixel location**. Two consequences follow immediately.

**CNOT permutes pixels.** CNOT is a permutation of computational basis states, so under amplitude encoding it relabels pixel positions. The permutation does not depend on $\theta$, so it cannot be made small — structure is destroyed at any parameter setting.

**$R_y$ rotates, it does not average.** On the amplitude pair $(c_a, c_b)$ of two basis states differing in qubit $q$:

$$\begin{pmatrix} c_a' \\ c_b' \end{pmatrix} = \begin{pmatrix} \cos\frac{\theta}{2} & -\sin\frac{\theta}{2} \\ \sin\frac{\theta}{2} & \cos\frac{\theta}{2} \end{pmatrix}\begin{pmatrix} c_a \\ c_b \end{pmatrix}$$

Pixel intensities are non-negative, so $c_a, c_b \ge 0$ and this produces a **difference** in one component and a **sum** in the other. At $\theta=\pi/2$ it is exactly a Haar wavelet step. A smoothing operator must be doubly stochastic on the intensity vector, and no unitary acting linearly on non-negative amplitudes has that property except the identity and permutations.

**Unitary evolution of amplitudes cannot implement diffusion.** Let us verify this rather than assert it.

In [ ]:
def old_style_ansatz(n_qubits, theta, layers=2):
    """The formulation this work replaces: Ry rotations + CNOT entangling ring."""
    qc = QuantumCircuit(n_qubits)
    for _ in range(layers):
        for q in range(n_qubits):
            qc.ry(theta, q)
        for q in range(n_qubits):
            qc.cx(q, (q + 1) % n_qubits)
    return qc


def apply_amplitude_ansatz(img, U, patch=8):
    """Amplitude-encode each patch, evolve, read out |amplitude|."""
    p = patch
    f = img.astype(np.float64)
    out = np.empty_like(f)
    for i in range(0, f.shape[0], p):
        for j in range(0, f.shape[1], p):
            blk = f[i:i+p, j:j+p].ravel()
            nrm = np.linalg.norm(blk)
            if nrm < 1e-12:
                out[i:i+p, j:j+p] = 0.0
                continue
            dec = np.abs(U @ (blk / nrm)) * nrm
            out[i:i+p, j:j+p] = dec.reshape(p, p)
    return np.clip(out, 0, 255).astype(np.uint8)


# A smooth test patch: a true smoothing operator should barely change it.
yy, xx = np.mgrid[0:64, 0:64]
smooth = np.clip(128 + 60 * np.sin(xx / 12.0), 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(smooth, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("input (already smooth)")
for ax, th in zip(axes[1:], [0.05, 0.20, 0.50]):
    U = Operator(old_style_ansatz(6, th)).data
    res = apply_amplitude_ansatz(smooth, U)
    ax.imshow(res, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"Ry/CNOT, theta={th}\nSSIM={ssim(smooth, res, data_range=255):.3f}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("A smoothing operator applied to an already-smooth image should be nearly")
print("the identity. Instead structure collapses even at theta = 0.05.")

## 3. QPAR done correctly: an ancilla-controlled shift channel

The fix has two parts.

**Encode intensity in probabilities, not amplitudes.** Set $c_r = \sqrt{I(r)/T}$ with $T=\sum_r I(r)$, so that $|\langle r|\psi\rangle|^2 = I(r)/T$. The image now lives in the *probabilities*, which a channel can mix convexly.

**Use a channel, not a unitary.** Introduce an ancilla prepared as $R_y(2\arcsin\sqrt{s})|0\rangle$, apply a **controlled cyclic incrementer** on the coordinate register, then trace out the ancilla:

$$\mathcal{E}_s(\rho) = (1-s)\rho + s\,S\rho S^\dagger \quad\Longrightarrow\quad p_{\text{out}}(r) = (1-s)\,p(r) + s\,p(r-e)$$

That is a genuine local average of strength $s$. One ancilla per axis gives a separable 2-D average. Total intensity is conserved exactly, by unitarity of the dilation.

This is physically realisable: it is a standard Stinespring dilation, not a mathematical convenience.

In [ ]:
def controlled_cyclic_increment(qc, reg, ctrl):
    """Controlled cyclic increment (mod 2^len(reg)) on `reg`."""
    for i in range(len(reg) - 1, 0, -1):
        qc.mcx([ctrl] + reg[:i], reg[i])
    qc.cx(ctrl, reg[0])


def build_qpar_circuit(k, s):
    """2k coordinate qubits + 2 ancillas. k = qubits per axis, patch = 2^k."""
    n = 2 * k
    qc = QuantumCircuit(n + 2)
    ax, ay = n, n + 1
    angle = 2.0 * np.arcsin(np.sqrt(s))
    qc.ry(angle, ax)
    qc.ry(angle, ay)
    controlled_cyclic_increment(qc, list(range(k)), ax)          # shift in x
    controlled_cyclic_increment(qc, list(range(k, 2 * k)), ay)   # shift in y
    return qc


qc_demo = build_qpar_circuit(k=3, s=0.25)   # 8x8 patch
print(qc_demo.draw(output="text"))
t = transpile(qc_demo, basis_gates=["u", "cx"], optimization_level=1)
print(f"\ntranspiled: depth {t.depth()}, {sum(t.count_ops().values())} gates, "
      f"{t.count_ops().get('cx', 0)} CX")

In [ ]:
class QPARChannel:
    """Patch-wise quantum smoothing channel.

    Encode I(r)/T in probabilities, evolve with both ancillas in |0>,
    trace out the ancillas, rescale by T.
    """

    def __init__(self, patch=8, s=0.25):
        assert (patch & (patch - 1)) == 0, "patch must be a power of two"
        assert 0.0 <= s <= 0.5, "mixing strength must lie in [0, 0.5]"
        self.patch, self.s = patch, s
        self.k = int(np.log2(patch))
        self.n_data = 2 * self.k
        self.n_qubits = self.n_data + 2
        self.dim = 2 ** self.n_data
        self.circuit = build_qpar_circuit(self.k, s)
        self.U = Operator(self.circuit).data
        t = transpile(self.circuit, basis_gates=["u", "cx"], optimization_level=1)
        self.depth = t.depth()
        self.gate_count = sum(t.count_ops().values())
        self.cx_count = t.count_ops().get("cx", 0)

    def process(self, img):
        p, D = self.patch, self.dim
        h, w = img.shape
        ph, pw = (p - h % p) % p, (p - w % p) % p
        f = np.pad(img.astype(np.float64), ((0, ph), (0, pw)), mode="reflect")
        out = np.empty_like(f)
        psi = np.zeros(D * 4)
        for i in range(0, f.shape[0], p):
            for j in range(0, f.shape[1], p):
                blk = f[i:i+p, j:j+p].ravel()
                total = blk.sum()
                if total < 1e-12:
                    out[i:i+p, j:j+p] = 0.0
                    continue
                psi[:] = 0.0
                psi[:D] = np.sqrt(blk / total)                     # encode
                phi = self.U @ psi                                 # evolve
                pr = (np.abs(phi) ** 2).reshape(4, D).sum(axis=0)  # trace out
                out[i:i+p, j:j+p] = (pr * total).reshape(p, p)
        return np.clip(out[:h, :w], 0, 255).astype(np.uint8)


qpar = QPARChannel(patch=8, s=0.25)
print(f"QPAR: {qpar.n_qubits} qubits ({qpar.n_data} data + 2 ancilla), "
      f"depth {qpar.depth}, {qpar.gate_count} gates, {qpar.cx_count} CX")

# Sanity check: does it actually smooth?
res = qpar.process(smooth)
print(f"SSIM on the smooth test image: {ssim(smooth, res, data_range=255):.4f}  "
      f"(the Ry/CNOT ansatz scored far below this)")

## 3b. Classical control: the box filter QPAR is equivalent to

Section 8 below proves QPAR is exactly a separable two-tap box filter. Rather than
only proving that on paper, we implement that same filter directly in NumPy --
no Qiskit, no state preparation, no circuit simulation, no measurement/decoding
step -- and benchmark it under the identical timing protocol as every other
method. This isolates how much of QPAR's speed-up is the quantum-simulator
machinery versus the underlying filter itself: any timing gap between this
control and Pure QPAR *is* that overhead, and by construction the two should
give statistically indistinguishable SSIM/PSNR (Section 8 confirms this to
within 8-bit rounding).


In [ ]:
class BoxFilterClassical:
    """The classical control for QPAR.

    Computes I_out = [(1-s)*Id + s*Sx] [(1-s)*Id + s*Sy] I directly with
    NumPy, patch by patch, with the same reflect-padding and per-patch
    cyclic (wrap-around) shift semantics as QPARChannel.process -- so the
    two are doing the identical operation, just one through a simulated
    quantum channel and one through plain array roll/multiply.
    """

    def __init__(self, patch=8, s=0.25):
        assert (patch & (patch - 1)) == 0, "patch must be a power of two"
        assert 0.0 <= s <= 0.5, "mixing strength must lie in [0, 0.5]"
        self.patch, self.s = patch, s

    def process(self, img):
        p, s = self.patch, self.s
        h, w = img.shape
        ph, pw = (p - h % p) % p, (p - w % p) % p
        f = np.pad(img.astype(np.float64), ((0, ph), (0, pw)), mode="reflect")
        out = np.empty_like(f)
        for i in range(0, f.shape[0], p):
            for j in range(0, f.shape[1], p):
                blk = f[i:i+p, j:j+p]
                bx = (1 - s) * blk + s * np.roll(blk, 1, axis=1)
                by = (1 - s) * bx + s * np.roll(bx, 1, axis=0)
                out[i:i+p, j:j+p] = by
        return np.clip(out[:h, :w], 0, 255).astype(np.uint8)


box_filter = BoxFilterClassical(patch=8, s=0.25)

# Sanity check against the quantum channel on the same test image used above
res_box = box_filter.process(smooth)
res_qpar_check = QPARChannel(patch=8, s=0.25).process(smooth)
print(f"max|box filter - QPAR| on test image: "
      f"{np.abs(res_box.astype(int) - res_qpar_check.astype(int)).max()} grey level(s)")


## 4. Paired clinical data and simulated low-dose degradation

The reference image is the acquired clinical chest radiograph, treated as a *surrogate* clean image. A paired low-dose observation is generated using Beer–Lambert transmission, Poisson photon noise (`N0=60`), Gaussian read-out noise (`sigma=4`), logarithmic transform and 8-bit quantization. No genuinely noise-free or acquired low-dose ground truth is claimed.

**Dataset is required** for the clinical benchmark. Set `CHEST_XRAY_DIR` (or `CHEST_DIR` below) to the local Kaggle Chest X-Ray Pneumonia directory. Missing files terminate with an explicit error, rather than substituting synthetic hand phantoms. The synthetic phantom generators are only for the separate phantom-control experiment later.


In [ ]:
import numpy as np

MU_SOFT, MU_TRAB, MU_CORT = 0.34, 0.62, 1.00


def _segdist(xx, yy, p0, p1):
    px, py = p1[0]-p0[0], p1[1]-p0[1]
    L2 = px*px + py*py
    if L2 < 1e-12:
        return np.hypot(xx-p0[0], yy-p0[1])
    t = np.clip(((xx-p0[0])*px + (yy-p0[1])*py)/L2, 0.0, 1.0)
    return np.hypot(xx-(p0[0]+t*px), yy-(p0[1]+t*py))


def _poly(xx, yy, pts):
    """Even-odd point-in-polygon on the grid."""
    inside = np.zeros(xx.shape, dtype=bool)
    n = len(pts)
    j = n-1
    for i in range(n):
        xi, yi = pts[i]; xj, yj = pts[j]
        cond = ((yi > yy) != (yj > yy)) & \
               (xx < (xj-xi)*(yy-yi)/(yj-yi+1e-12)+xi)
        inside ^= cond
        j = i
    return inside


def _bone(img, xx, yy, p0, p1, r, epi=1.18, cort=0.30):
    """Diaphysis capsule with bulbous epiphyses at both ends."""
    d = _segdist(xx, yy, p0, p1)
    shaft = d <= r
    img[shaft] = np.maximum(img[shaft], MU_CORT)
    core = d <= r*(1-cort)
    img[core] = MU_TRAB
    for e in (p0, p1):
        de = np.hypot(xx-e[0], yy-e[1])
        head = de <= r*epi
        img[head] = np.maximum(img[head], MU_CORT)
        img[de <= r*epi*(1-cort*0.8)] = MU_TRAB


def hand_phantom(size, rng):
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float64)
    img = np.zeros((size, size))
    S = size/256.0
    sc = S*rng.uniform(0.94, 1.04)
    cx = size*0.5 + rng.normal(0, size*0.012)
    wrist = size*0.93
    C = (cx, wrist)
    tilt = np.deg2rad(rng.normal(0, 4))

    def R(p):
        ca, sa = np.cos(tilt), np.sin(tilt)
        dx, dy = p[0]-C[0], p[1]-C[1]
        return (C[0]+ca*dx-sa*dy, C[1]+sa*dx+ca*dy)

    # ---------- geometry ----------
    palm_h = 62*sc
    knuck_y = wrist - palm_h
    wrist_hw = 20*sc          # half width at wrist
    knuck_hw = 30*sc          # half width at knuckles

    # knuckle x positions (index..little), MCP row arches slightly
    kx = [cx + d*sc for d in (-21, -7, 7, 20)]
    ky = [knuck_y + d*sc for d in (4, -2, 0, 7)]
    # finger splay and length multiplier
    spl = [-9, -3, 3, 10]
    flen = [0.95, 1.04, 0.98, 0.80]

    # ---------- soft tissue ----------
    # forearm
    d = _segdist(xx, yy, R((cx, size*1.06)), R((cx, wrist-6*sc)))
    img[d <= 22*sc] = MU_SOFT
    # palm outline
    palm = [R((cx-wrist_hw, wrist)), R((cx-knuck_hw, knuck_y+6*sc)),
            R((cx-knuck_hw*0.9, knuck_y-4*sc)), R((cx+knuck_hw*0.9, knuck_y-4*sc)),
            R((cx+knuck_hw, knuck_y+8*sc)), R((cx+wrist_hw, wrist))]
    img[_poly(xx, yy, palm)] = MU_SOFT
    # thenar eminence (thumb muscle bulge)
    d = _segdist(xx, yy, R((cx-16*sc, wrist-8*sc)), R((cx-24*sc, knuck_y+22*sc)))
    img[d <= 13*sc] = MU_SOFT

    # ---------- digit soft tissue + bones ----------
    joints = []
    for k in range(4):
        a = np.deg2rad(spl[k] + rng.normal(0, 2.5))
        fl = flen[k]*rng.uniform(0.95, 1.05)
        cur = (kx[k], ky[k])
        segs = []
        for ln, r in ((26, 3.4), (17, 2.9), (11, 2.4)):
            ln = ln*sc*fl*rng.uniform(0.94, 1.06)
            nxt = (cur[0]+np.sin(a)*ln, cur[1]-np.cos(a)*ln)
            segs.append((cur, nxt, r*sc*rng.uniform(0.93, 1.07)))
            cur = (nxt[0]+np.sin(a)*2.2*sc, nxt[1]-np.cos(a)*2.2*sc)
            a += np.deg2rad(rng.normal(0, 2))
        # soft sleeve first
        for p0, p1, r in segs:
            d = _segdist(xx, yy, R(p0), R(p1))
            img[d <= r+4.2*sc] = np.maximum(img[d <= r+4.2*sc], MU_SOFT)
        # fingertip pad
        d = np.hypot(xx-R(segs[-1][1])[0], yy-R(segs[-1][1])[1])
        img[d <= segs[-1][2]+4.4*sc] = np.maximum(img[d <= segs[-1][2]+4.4*sc], MU_SOFT)
        joints.append(segs)

    # thumb
    ta = np.deg2rad(-52 + rng.normal(0, 4))
    tcur = (cx-17*sc, wrist-24*sc)
    tsegs = []
    for ln, r in ((30, 4.2), (20, 3.5), (13, 2.9)):
        ln = ln*sc*rng.uniform(0.94, 1.06)
        nxt = (tcur[0]+np.sin(ta)*ln, tcur[1]-np.cos(ta)*ln)
        tsegs.append((tcur, nxt, r*sc))
        tcur = (nxt[0]+np.sin(ta)*2.4*sc, nxt[1]-np.cos(ta)*2.4*sc)
        ta += np.deg2rad(rng.normal(0, 4)+8)
    for p0, p1, r in tsegs:
        d = _segdist(xx, yy, R(p0), R(p1))
        img[d <= r+4.6*sc] = np.maximum(img[d <= r+4.6*sc], MU_SOFT)
    d = np.hypot(xx-R(tsegs[-1][1])[0], yy-R(tsegs[-1][1])[1])
    img[d <= tsegs[-1][2]+4.6*sc] = np.maximum(img[d <= tsegs[-1][2]+4.6*sc], MU_SOFT)

    # ---------- bones ----------
    # radius + ulna
    for dx, r in ((-9.5, 7.0), (8.0, 5.6)):
        _bone(img, xx, yy, R((cx+dx*sc, size*1.05)),
              R((cx+dx*0.9*sc, wrist-10*sc)), r*sc, epi=1.15)
    # carpals: two rows of 4
    for row, (ry, n) in enumerate(((wrist-16*sc, 4), (wrist-26*sc, 4))):
        for i in range(n):
            bx = cx + (i-(n-1)/2)*9.5*sc + rng.normal(0, 1.2*sc)
            by = ry + rng.normal(0, 1.5*sc)
            p = R((bx, by))
            _bone(img, xx, yy, p, p, rng.uniform(3.4, 4.5)*sc, epi=1.0, cort=0.26)
    # metacarpals: from carpal row out to each knuckle
    for k in range(4):
        base = (cx+(k-1.5)*8.0*sc, wrist-30*sc)
        _bone(img, xx, yy, R(base), R((kx[k], ky[k])), 3.6*sc, epi=1.16)
    # thumb metacarpal + phalanges
    for p0, p1, r in tsegs:
        _bone(img, xx, yy, R(p0), R(p1), r, epi=1.14)
    # finger phalanges
    for segs in joints:
        for p0, p1, r in segs:
            _bone(img, xx, yy, R(p0), R(p1), r, epi=1.16)

    img = np.clip(img, 0, 1)
    return (img*255).astype(np.uint8)


import numpy as np


def _rot(xx, yy, cx, cy, ang):
    ca, sa = np.cos(ang), np.sin(ang)
    return (cx + ca * (xx - cx) + sa * (yy - cy),
            cy - sa * (xx - cx) + ca * (yy - cy))


def _stamp(img, val, px, py, rad, mask=None):
    """Rasterise a polyline (px,py) as a thick stroke via a disk stamp.

    Much faster than computing a full-image distance per sample point.
    """
    h, w = img.shape
    r = int(np.ceil(rad))
    dy, dx = np.mgrid[-r:r + 1, -r:r + 1]
    disk = dx * dx + dy * dy <= rad * rad
    oy, ox = np.where(disk)
    oy -= r; ox -= r
    xi = np.round(px).astype(int)
    yi = np.round(py).astype(int)
    for x0, y0 in zip(xi, yi):
        ys = y0 + oy; xs = x0 + ox
        ok = (ys >= 0) & (ys < h) & (xs >= 0) & (xs < w)
        ys, xs = ys[ok], xs[ok]
        if mask is not None:
            keep = mask[ys, xs]
            ys, xs = ys[keep], xs[keep]
        cur = img[ys, xs]
        img[ys, xs] = np.where(cur < val, val, cur)



def chest_phantom(size, rng):
    """Return a randomised PA chest radiograph attenuation map as uint8."""
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float64)
    S = size / 256.0
    img = np.zeros((size, size), dtype=np.float64)

    cx = size * 0.5 + rng.normal(0, size * 0.015)
    cy = size * 0.52 + rng.normal(0, size * 0.02)
    ang = np.deg2rad(rng.normal(0, 3))
    xr, yr = _rot(xx, yy, cx, cy, ang)

    half_w = size * rng.uniform(0.30, 0.35)      # thorax half-width
    half_h = size * rng.uniform(0.34, 0.40)      # thorax half-height

    mu_soft = rng.uniform(0.24, 0.32)
    mu_lung = rng.uniform(0.06, 0.12)
    mu_medi = rng.uniform(0.42, 0.52)
    mu_bone = rng.uniform(0.85, 1.00)

    # ---- soft-tissue body outline (rounded thorax) ----------------------
    body = ((xr - cx) / half_w) ** 2 + ((yr - cy) / half_h) ** 2 <= 1.0
    img[body] = mu_soft
    # apical soft tissue: widen the top of the thorax into the shoulders
    apex = ((xr - cx) / (half_w * 1.02)) ** 2 + \
           ((yr - (cy - half_h * 0.55)) / (half_h * 0.42)) ** 2 <= 1.0
    img[apex & (yr < cy - half_h * 0.45)] = mu_soft

    # ---- lung fields (two paramediastinal ellipses, darker) -------------
    lung_top = cy - half_h * 0.55
    lung_h = half_h * rng.uniform(0.72, 0.85)
    lung_w = half_w * rng.uniform(0.34, 0.42)
    dia = cy + half_h * rng.uniform(0.30, 0.45)      # diaphragm level
    for sx in (-1, 1):
        lx = cx + sx * half_w * 0.42
        e = ((xr - lx) / lung_w) ** 2 + ((yr - (lung_top + lung_h * 0.5))
                                         / (lung_h * 0.5)) ** 2 <= 1.0
        lung = e & body & (yr < dia)
        img[lung] = mu_lung

    # ---- mediastinum + heart shadow -------------------------------------
    medi_w = half_w * rng.uniform(0.16, 0.22)
    medi = (np.abs(xr - cx) < medi_w) & (yr > cy - half_h * 0.55) & (yr < dia)
    img[medi] = mu_medi
    heart_side = rng.choice([-1, -1, -1, 1])          # usually left (image L)
    hx = cx + heart_side * half_w * rng.uniform(0.14, 0.22)
    hy = cy + half_h * rng.uniform(0.10, 0.25)
    hr_w = half_w * rng.uniform(0.24, 0.32)
    hr_h = half_h * rng.uniform(0.24, 0.32)
    heart = ((xr - hx) / hr_w) ** 2 + ((yr - hy) / hr_h) ** 2 <= 1.0
    img[heart & (yr < dia)] = mu_medi

    # ---- diaphragm domes (soft tissue below the lungs) ------------------
    img[body & (yr >= dia)] = mu_soft

    # ---- thoracic spine (vertical column of vertebrae) ------------------
    spine_w = half_w * 0.10
    col = (np.abs(xr - cx) < spine_w) & (yr > cy - half_h * 0.7) & (yr < dia + half_h * 0.2)
    img[col] = np.maximum(img[col], mu_bone * 0.85)
    n_vert = int(rng.integers(9, 12))
    for k in range(n_vert):
        vy = (cy - half_h * 0.6) + k * (half_h * 1.05) / n_vert
        d = np.hypot((xr - cx) / (spine_w * 1.3), (yr - vy) / (half_h * 0.028))
        img[d <= 1.0] = mu_bone

    # ---- rib cage: symmetric arcs springing from the spine ---------------
    n_ribs = int(rng.integers(9, 11))
    rib_thick = max(1.0, half_h * 0.016 * rng.uniform(0.9, 1.15))
    lung_bottom = dia
    for k in range(n_ribs):
        frac = k / (n_ribs - 1)                       # 0 (top) .. 1 (bottom)
        y_spine = cy - half_h * 0.52 + frac * half_h * 0.95   # rib origin on spine
        span = np.deg2rad(rng.uniform(150, 172))      # arc angular length
        droop = 0.55 + 0.5 * frac                     # lower ribs droop more
        rw = half_w * (0.55 + 0.42 * np.sin(np.pi * (0.35 + 0.55 * frac)))
        rh = half_h * (0.10 + 0.20 * frac)
        for sx in (-1, 1):
            t = np.linspace(0, 1, 200)
            phi = t * span
            px = cx + sx * rw * np.sin(phi)
            py = y_spine + rh * (1 - np.cos(phi)) * droop
            keep = py <= lung_bottom + half_h * 0.05
            _stamp(img, mu_bone * rng.uniform(0.80, 0.92), px[keep], py[keep],
                   rib_thick, mask=body)

    # ---- clavicles (two shallow arcs near the top) ----------------------
    for sx in (-1, 1):
        t = np.linspace(0, 1, 120)
        px = cx + sx * half_w * 0.95 * t
        py = cy - half_h * 0.66 + half_h * 0.06 * np.sin(t * np.pi)
        _stamp(img, mu_bone, px, py, rib_thick * 1.4, mask=body)

    img = np.clip(img, 0.0, 1.0)
    return (img * 255).astype(np.uint8)


PHOTON_COUNT, READ_NOISE = 60.0, 4.0


def xray_degrade(clean, rng, photon_count=PHOTON_COUNT, read_noise=READ_NOISE):
    """Low-dose X-ray acquisition model applied to a clean image."""
    mu = clean.astype(np.float64) / 255.0
    trans = np.exp(-2.5 * mu)                                   # Beer-Lambert
    detected = rng.poisson(photon_count * trans).astype(np.float64)
    detected += rng.normal(0.0, read_noise, detected.shape)     # read-out noise
    detected = np.clip(detected, 1.0, None)
    proj = -np.log(detected / photon_count)                     # log transform
    lo, hi = np.percentile(proj, [0.5, 99.5])
    img = np.clip((proj - lo) / max(hi - lo, 1e-9), 0.0, 1.0)
    return (img * 255.0).astype(np.uint8)


# ---------------------------------------------------------------------------
# CLINICAL DATASET (required for Figures 1--3 and Table I).
# Point CHEST_DIR to the actual Kaggle chest_xray folder on YOUR computer.
# You can instead set the environment variable CHEST_XRAY_DIR.
# IMPORTANT: Do not silently substitute synthetic phantoms when the
# clinical dataset is missing: doing so makes Figure 3 scientifically invalid.
# ---------------------------------------------------------------------------
from pathlib import Path

CHEST_DIR = os.environ.get("CHEST_XRAY_DIR", "data/chest_xray")
CHEST_DIR = Path(CHEST_DIR).expanduser()
SIZE, N_IMAGES = 128, 5856

def square_resize(img, size):
    h, w = img.shape
    side = min(h, w)
    img = img[(h-side)//2:(h-side)//2+side, (w-side)//2:(w-side)//2+side]
    interp = cv2.INTER_AREA if side > size else cv2.INTER_CUBIC
    return cv2.resize(img, (size, size), interpolation=interp)

print("Notebook working directory:", Path.cwd())
print("Clinical dataset search path:", CHEST_DIR.resolve())
if not CHEST_DIR.is_dir():
    raise FileNotFoundError(
        f"Clinical chest X-ray dataset not found: {CHEST_DIR.resolve()}\n"
        "Download/extract the Chest X-Ray Pneumonia dataset and change "
        "CHEST_DIR above to the folder containing train/test/val "
        "(directly or in a nested chest_xray subfolder). "
        "Do not bypass this check by relabelling phantom images."
    )
paths = sorted(str(p) for p in CHEST_DIR.rglob("*.jpeg"))
if len(paths) < N_IMAGES:
    raise RuntimeError(
        f"Found only {len(paths)} .jpeg files under {CHEST_DIR.resolve()}, "
        f"but {N_IMAGES} clinical images are required. Verify the dataset."
    )
if len(paths) != N_IMAGES:
    print(f"ATTENTION: Found {len(paths)} images, sampling {N_IMAGES} "
          "with fixed seed as in the original notebook.")

rng = np.random.default_rng(42)
idx = sorted(rng.choice(len(paths), size=N_IMAGES, replace=False))
loaded = [cv2.imread(paths[i], cv2.IMREAD_GRAYSCALE) for i in idx]
if any(img is None for img in loaded):
    raise RuntimeError("At least one clinical JPEG cannot be decoded.")
refs = np.stack([square_resize(img, SIZE) for img in loaded])
obs = np.stack([xray_degrade(r, rng) for r in refs])
DATASET = "chest radiographs (real, %d of %d)" % (len(refs), len(paths))
print(f"{len(refs)} clinical pairs at {SIZE}x{SIZE} -- {DATASET}")

fig, axes = plt.subplots(2, 5, figsize=(11, 4.6))
for c_ in range(5):
    axes[0, c_].imshow(refs[c_], cmap="gray", vmin=0, vmax=255)
    axes[0, c_].axis("off")
    axes[1, c_].imshow(obs[c_], cmap="gray", vmin=0, vmax=255)
    axes[1, c_].axis("off")
axes[0, 2].set_title("clinical references (top) and simulated low-dose observations (bottom)")
plt.tight_layout()
plt.show()


## 5. Metrics

Contrast enhancement **deliberately** changes the global intensity mapping, so raw PSNR/SSIM punish a method for doing its job. We report **intensity-matched** metrics: remap the output by the least-squares affine fit $aI_{\text{out}}+b$ to the reference before scoring. This removes a nuisance degree of freedom and isolates structural fidelity.

We add entropy, EME (Measure of Enhancement), and RMS contrast as no-reference indicators of diagnostic visibility.

In [ ]:
def eme(img, block=8):
    """Measure of Enhancement: mean local log contrast ratio (no-reference)."""
    f = img.astype(np.float64)
    vals = [20.0 * np.log10((f[i:i+block, j:j+block].max() + 1.0) /
                            (f[i:i+block, j:j+block].min() + 1.0))
            for i in range(0, f.shape[0] - block + 1, block)
            for j in range(0, f.shape[1] - block + 1, block)]
    return float(np.mean(vals)) if vals else 0.0


def intensity_match(out, ref):
    """Least-squares affine remap of `out` onto the range of `ref`."""
    x = out.astype(np.float64).ravel()
    y = ref.astype(np.float64).ravel()
    if x.var() < 1e-9:
        return out.astype(np.uint8)
    a = np.cov(x, y, bias=True)[0, 1] / x.var()
    b = y.mean() - a * x.mean()
    return np.clip(a * out.astype(np.float64) + b, 0, 255).astype(np.uint8)


def evaluate(ref, out):
    m = intensity_match(out, ref)
    return {"ssim": float(ssim(ref, m, data_range=255)),
            "psnr": float(psnr(ref, m, data_range=255)),
            "ssim_raw": float(ssim(ref, out, data_range=255)),
            "entropy": float(shannon_entropy(out)),
            "eme": eme(out),
            "rms_contrast": float(out.astype(np.float64).std())}

## 6. The four pipelines

Note the **ordering** of the two hybrids — it turns out to matter a great deal.

In [ ]:
def method_conventional(img):
    """Classical baseline: Non-Local Means denoising, then CLAHE."""
    t0 = time.perf_counter()
    den = cv2.fastNlMeansDenoising(img, None, h=10,
                                   templateWindowSize=7, searchWindowSize=21)
    res = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8)).apply(den)
    return res, time.perf_counter() - t0


def method_quantum_pure(img, qpar):
    t0 = time.perf_counter()
    res = qpar.process(img)
    return res, time.perf_counter() - t0


def method_hybrid(img, qpar):
    """Proposed: quantum smoothing FIRST, then classical contrast."""
    t0 = time.perf_counter()
    sm = qpar.process(img)
    res = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(2, 2)).apply(sm)
    return res, time.perf_counter() - t0


def method_hybrid_reversed(img, qpar):
    """Ablation: CLAHE first, then QPAR (the ordering used previously)."""
    t0 = time.perf_counter()
    pre = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img)
    res = qpar.process(pre)
    return res, time.perf_counter() - t0

def method_box_filter_classical(img, box):
    """Classical control: the box filter QPAR is provably equivalent to
    (Sec. III-C of the paper), run with no quantum simulator involved."""
    t0 = time.perf_counter()
    res = box.process(img)
    return res, time.perf_counter() - t0


## 7. Benchmark over 5856 paired images

In [ ]:
# Full clinical benchmark: only run after the dataset validation cell.
if not str(globals().get("DATASET", "")).startswith("chest radiographs (real"):
    raise RuntimeError("Clinical benchmark requires actual chest radiographs.")
methods = {
    "conventional": lambda im: method_conventional(im),
    "quantum_pure": lambda im: method_quantum_pure(im, qpar),
    "box_filter": lambda im: method_box_filter_classical(im, box_filter),
    "hybrid": lambda im: method_hybrid(im, qpar),
    "hybrid_reversed": lambda im: method_hybrid_reversed(im, qpar),
}
KEYS = ["time", "ssim", "psnr", "entropy", "eme", "rms_contrast"]
acc = {m: {k: [] for k in KEYS} for m in methods}
acc["degraded_input"] = {k: [] for k in KEYS}
samples = None

for n, (ref, ob) in enumerate(zip(refs, obs)):
    base = evaluate(ref, ob)
    acc["degraded_input"]["time"].append(0.0)
    for k in KEYS[1:]:
        acc["degraded_input"][k].append(base[k])
    outs = {}
    for name, fn in methods.items():
        res, dt = fn(ob)
        met = evaluate(ref, res)
        acc[name]["time"].append(dt)
        for k in KEYS[1:]:
            acc[name][k].append(met[k])
        outs[name] = res
    if n == 0:
        samples = {"reference": ref, "degraded": ob, **outs}
    if (n + 1) % 250 == 0:
        print(f"  {n+1}/{len(refs)}")

summary = {m: {k: (float(np.mean(v)), float(np.std(v))) for k, v in d.items()}
           for m, d in acc.items()}

ORDER = ["degraded_input", "conventional", "quantum_pure", "box_filter", "hybrid", "hybrid_reversed"]
print(f"\n{'method':<18}{'time(ms)':>10}{'SSIM':>9}{'PSNR':>9}{'entropy':>9}{'EME':>8}")
for m in ORDER:
    d = summary[m]
    print(f"{m:<18}{d['time'][0]*1e3:>10.2f}{d['ssim'][0]:>9.3f}"
          f"{d['psnr'][0]:>9.2f}{d['entropy'][0]:>9.2f}{d['eme'][0]:>8.1f}")

### Reproducibility audit: export original per-image measurements

Run this cell after the benchmark, before closing the kernel. It preserves the paired observations needed to reproduce confidence intervals and tabular results without rerunning every pipeline.


In [ ]:
# Save the ORIGINAL measured per-image outputs (not fabricated summary data).
# Run this immediately after the clinical benchmark cell, while acc is in memory.
import csv, json, platform, sys
from pathlib import Path
outdir = Path("results")
outdir.mkdir(exist_ok=True)
metric_order = ["time", "ssim", "psnr", "entropy", "eme", "rms_contrast"]
all_methods = ["degraded_input", "conventional", "quantum_pure", "box_filter", "hybrid", "hybrid_reversed"]
if not all(m in acc for m in all_methods):
    raise RuntimeError("Run the clinical benchmark cell first; acc is missing.")
num = len(acc["conventional"]["ssim"])
if len(refs) != num:
    raise RuntimeError("Clinical benchmark size mismatch: refs and acc differ.")
with (outdir / "clinical_per_image_metrics.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["image_index", "method"] + metric_order)
    for i in range(num):
        for m in all_methods:
            writer.writerow([i, m] + [acc[m][k][i] for k in metric_order])
with (outdir / "clinical_summary.json").open("w", encoding="utf-8") as f:
    json.dump({"dataset": DATASET, "n": num, "summary": summary}, f, indent=2)
import qiskit
manifest = {"python": sys.version, "platform": platform.platform(),
            "qiskit": qiskit.__version__, "numpy": np.__version__,
            "opencv": cv2.__version__, "opencv_threads": cv2.getNumThreads(),
            "method": "Qiskit Operator dense unitary + NumPy statevector multiplication",
            "dataset": DATASET, "n": num}
with (outdir / "execution_manifest.json").open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print("Saved per-image clinical metrics, summaries, and execution manifest in results/")
print(json.dumps(manifest, indent=2))


In [ ]:
panels = [("reference", "Ground truth"), ("degraded", "Degraded"),
          ("conventional", "Conventional"), ("quantum_pure", "Pure QPAR"),
          ("hybrid", "Hybrid"), ("hybrid_reversed", "Hybrid (rev.)")]
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
for ax, (k, t) in zip(axes, panels):
    ax.imshow(samples[k], cmap="gray", vmin=0, vmax=255)
    ax.set_title(t, fontsize=10); ax.axis("off")
os.makedirs("figures", exist_ok=True)
plt.tight_layout()
fig.savefig("figures/fig_qualitative.pdf", bbox_inches="tight")
plt.show()
print("Saved figures/fig_qualitative.pdf")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
colors = {"degraded_input": "0.55", "conventional": "#b2182b",
          "quantum_pure": "#2166ac", "box_filter": "#f4a582",
          "hybrid": "#1a9850", "hybrid_reversed": "#7b3294"}
labels = {"degraded_input": "Degraded", "conventional": "Conventional",
          "quantum_pure": "Pure QPAR", "box_filter": "Box filter (classical)",
          "hybrid": "Hybrid", "hybrid_reversed": "Hybrid (rev.)"}

for ax, metric, name in [(axes[0], "ssim", "SSIM"), (axes[1], "psnr", "PSNR (dB)")]:
    vals = [summary[m][metric][0] for m in ORDER]
    errs = [summary[m][metric][1] for m in ORDER]
    ax.bar(range(len(ORDER)), vals, yerr=errs, capsize=3,
           color=[colors[m] for m in ORDER], edgecolor="black", lw=0.5)
    ax.set_xticks(range(len(ORDER)))
    ax.set_xticklabels([labels[m] for m in ORDER], rotation=25, ha="right")
    ax.set_ylabel(name + " (higher is better)")
    ax.grid(axis="y", ls=":", alpha=0.5)
    ax.set_axisbelow(True)

ms = [m for m in ORDER if m != "degraded_input"]
axes[2].bar(range(len(ms)), [summary[m]["time"][0]*1e3 for m in ms],
            color=[colors[m] for m in ms], edgecolor="black", lw=0.5)
axes[2].set_xticks(range(len(ms)))
axes[2].set_xticklabels([labels[m] for m in ms], rotation=25, ha="right")
axes[2].set_ylabel("measured runtime (ms, lower is better)")
axes[2].grid(axis="y", ls=":", alpha=0.5); axes[2].set_axisbelow(True)
plt.tight_layout(); plt.show()

# --- computation time: the headline claim, tested properly ----------------
from scipy import stats
c_t = np.array(acc["conventional"]["time"]) * 1e3
q_t = np.array(acc["quantum_pure"]["time"]) * 1e3
b_t = np.array(acc["box_filter"]["time"]) * 1e3
h_t = np.array(acc["hybrid"]["time"]) * 1e3

boot_rng = np.random.default_rng(0)
def boot_ci(a, b, n=4000):
    r = [a[i].mean() / b[i].mean()
         for i in (boot_rng.integers(0, len(a), len(a)) for _ in range(n))]
    return np.percentile(r, [2.5, 97.5])

for nm, arr in [("Pure QPAR", q_t), ("Box filter", b_t), ("Hybrid", h_t)]:
    lo, hi = boot_ci(c_t, arr)
    p = stats.wilcoxon(arr, c_t).pvalue
    ratio = c_t.mean() / arr.mean()  # <1 means slower than conventional
    speed_text = (f"{ratio:.2f}x as fast" if ratio >= 1
                  else f"{1/ratio:.2f}x slower")
    print(f"{nm:<12} {arr.mean():.3f} ms | {1000/arr.mean():.0f} img/s | "
          f"{speed_text} vs conventional [speed ratio CI {lo:.2f}-{hi:.2f}], p={p:.2g}")
print(f"{'Conventional':<12} {c_t.mean():.3f} ms  |  {1000/c_t.mean():.0f} img/s")

d = h_t.mean() - q_t.mean()
pct = 100 * d / q_t.mean()
p_hq = stats.wilcoxon(h_t, q_t).pvalue
print(f"\nHybrid vs pure QPAR: {d:+.3f} ms ({pct:+.1f}%), p={p_hq:.2g}")
sig = "statistically detectable" if p_hq < 0.05 else "not statistically detectable"
print(f"{sig}, and only {abs(pct):.1f}% of runtime -- CLAHE is nearly free.")
print("The 0.4% overhead of CLAHE over simulated QPAR is small.")
print("Both quantum-emulated variants are slower than conventional in this run.")

# --- the number the paper needs: how much of QPAR's speed is simulator
# overhead versus the underlying filter (Sec. V-A / Sec. VI-A of the paper)
d_qb = q_t.mean() - b_t.mean()
ratio_qb = q_t.mean() / b_t.mean()
p_qb = stats.wilcoxon(q_t, b_t).pvalue
qual_gap = abs(summary["quantum_pure"]["ssim"][0] - summary["box_filter"]["ssim"][0])
print(f"\nPure QPAR vs classical box filter (identical operation, Sec. III-C): "
      f"QPAR is {d_qb:+.3f} ms slower ({ratio_qb:.2f}x), p={p_qb:.2g}. "
      f"This gap is state prep + circuit simulation + decoding overhead -- "
      f"paste it into the paper's Sec. V-A placeholder. "
      f"SSIM gap between the two: {qual_gap:.4f} (should be ~0, confirming "
      f"they compute the same thing).")
print("\nThe direct classical equivalent is faster; this is NOT quantum advantage.")

### Recompute paired confidence intervals from current per-image observations

After the clinical benchmark, the arrays `acc[...]` contain paired observations. The cell below computes percentile bootstrap 95% CIs for the hybrid-minus-conventional SSIM and PSNR mean differences (4,000 bootstrap resamples). These figures are recalculated from the local run, **not** from the rounded aggregate metrics in this repository.

In [ ]:
# Recompute paired clinical SSIM/PSNR intervals from the ACTUAL 5856 stored pairs.
# No bootstrap inference can be recovered from mean +/- SD alone.
from scipy import stats
rng_ci = np.random.default_rng(20261022)
def paired_bootstrap(a, b, n_boot=4000):
    x = np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64)
    means = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        means[i] = x[rng_ci.integers(0, len(x), len(x))].mean()
    return x.mean(), np.percentile(means, [2.5, 97.5]), stats.wilcoxon(x).pvalue, float(np.mean(x > 0))
for key in ('ssim', 'psnr'):
    effect, ci, pval, winfrac = paired_bootstrap(acc['hybrid'][key], acc['conventional'][key])
    print(f"Hybrid - conventional {key}: {effect:+.6f}; 95% CI [{ci[0]:+.6f}, {ci[1]:+.6f}]; p={pval:.4g}; wins={winfrac:.2%}")
# Percentile bootstrap with 4000 resamples, as indicated in the manuscript.


### Generate corrected runtime Figure 2 directly from recorded means

The figure is generated from `summary`, not hand-entered historical numbers.


In [ ]:
# Regenerate the horizontal runtime graphic from freshly measured results.
# No run-time numbers are hard-coded or claimed as universal benchmarks.
from pathlib import Path
Path("figures").mkdir(exist_ok=True)
labels_runtime = ["Conventional", "Pure QPAR", "Hybrid", "Classical QPAR"]
keys_runtime = ["conventional", "quantum_pure", "hybrid", "box_filter"]
means_ms = [1000.0 * summary[k]["time"][0] for k in keys_runtime]
base = means_ms[0]
fig, ax = plt.subplots(figsize=(6.0, 2.8))
positions = np.arange(len(labels_runtime))
ax.barh(positions, means_ms, color=["0.55", "0.7", "0.7", "0.83"], edgecolor="0.3")
for i, v in enumerate(means_ms):
    ax.text(v+max(means_ms)*0.015, i, f"{v:.2f} ms ({base/v:.2f}x)", va="center", fontsize=8)
ax.invert_yaxis()
ax.set_yticks(positions, labels_runtime)
ax.set_xlim(0, max(means_ms) * 1.40)
ax.set_xlabel("Measured mean per-image runtime (ms)")
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
fig.savefig("figures/fig_runtime.pdf", bbox_inches="tight")
plt.close(fig)
print("Saved figures/fig_runtime.pdf from the locally measured clinical benchmark.")


## 8. The cautionary result: QPAR is a box filter

The channel acts on probabilities as $p_{\text{out}}(r) = (1-s)p(r) + s\,p(r-e)$ per axis. That is a two-tap filter, and the two-axis composition is separable. So the whole thing is exactly

$$I_{\text{out}} = \big[(1-s)\mathbb{I} + sS_x\big]\big[(1-s)\mathbb{I} + sS_y\big]\,I$$

with $S_x, S_y$ cyclic shifts within the patch — computable classically in $O(1)$ per pixel. Let us verify it numerically rather than take it on faith.

In [ ]:
test = (np.random.default_rng(1).random((64, 64)) * 255).astype(np.uint8)

print("Verifying QPAR == separable two-tap box filter:\n")
for s in [0.10, 0.25, 0.40]:
    q = QPARChannel(patch=8, s=s)
    quantum = q.process(test).astype(np.float64)

    p, f = 8, test.astype(np.float64)
    classical = np.empty((64, 64))
    for i in range(0, 64, p):
        for j in range(0, 64, p):
            b = f[i:i+p, j:j+p]
            bx = (1 - s) * b + s * np.roll(b, 1, axis=1)     # cyclic shift x
            classical[i:i+p, j:j+p] = (1 - s) * bx + s * np.roll(bx, 1, axis=0)
    classical = np.round(np.clip(classical, 0, 255))

    print(f"  s={s:.2f}   max|quantum - classical| = "
          f"{np.abs(quantum - classical).max():.1f} grey level(s)")

print("\nAgreement is within 8-bit rounding. The circuit computes a box filter.")

**Interpretation of the stored benchmark.**

The QPAR-equivalent separable filter is **4.75x faster** than the conventional NLM--CLAHE pipeline, but dense-unitary QPAR emulation is **1.68x slower** than conventional and **7.96x slower** than its own direct classical equivalent. A speed-up attributable to quantum computation has not been demonstrated. The hybrid has a modest SSIM advantage over the selected classical baseline, but its runtime is worse and the benefit has not been clinically validated.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for m in ms:
    axes[0].scatter(summary[m]["time"][0]*1e3, summary[m]["ssim"][0], s=70,
                    color=colors[m], edgecolor="black", lw=0.6, label=labels[m], zorder=3)
    axes[1].scatter(summary[m]["ssim"][0], summary[m]["entropy"][0], s=70,
                    color=colors[m], edgecolor="black", lw=0.6, zorder=3)
    axes[1].annotate(labels[m], (summary[m]["ssim"][0], summary[m]["entropy"][0]),
                     textcoords="offset points", xytext=(7, -3), fontsize=8)
axes[0].axhline(summary["degraded_input"]["ssim"][0], ls="--", lw=0.8, color="0.4")
axes[0].text(0.98, summary["degraded_input"]["ssim"][0], "unprocessed", ha="right",
             va="bottom", fontsize=8, color="0.4",
             transform=axes[0].get_yaxis_transform())
axes[0].set_xlabel("runtime per image (ms)"); axes[0].set_ylabel("SSIM")
axes[0].legend(frameon=False, loc="lower right"); axes[0].grid(ls=":", alpha=0.5)
axes[1].set_xlabel("structural fidelity (SSIM)")
axes[1].set_ylabel("information content (entropy, bits)")
axes[1].grid(ls=":", alpha=0.5)
plt.tight_layout(); plt.show()

## 9. Phantom controls

The chest-anatomy and hand-phantom generators are both included. Set `RUN_PHANTOM_CONTROL = True` in the following cell to rerun the phantom experiments; it defaults to `False` because the full phantom benchmark is computationally and memory intensive.

Original executed notebook output reported the expected ranking reversal; see `../results/recorded_aggregate_metrics.csv`. Printed phantom metrics do not perfectly agree with one earlier manuscript version. Use the original data and full-precision summaries when making scientific claims.


In [ ]:
RUN_PHANTOM_CONTROL = False
N_PHANTOM = 5856  # same sample count as the original reported phantom run

if RUN_PHANTOM_CONTROL:
    rng_p = np.random.default_rng(42)
    refs_p = np.stack([chest_phantom(SIZE, rng_p) for _ in range(N_PHANTOM)])
    obs_p = np.stack([xray_degrade(r, rng_p) for r in refs_p])

    print(f"Anatomy-matched CHEST phantom control (n={N_PHANTOM})")
    print(f"{'method':<24}{'time(ms)':>10}{'SSIM':>9}{'PSNR':>9}")
    chest_phantom_summary = {}
    for name, fn in methods.items():
        times, ssims, psnrs = [], [], []
        for r, o in zip(refs_p, obs_p):
            res, dt = fn(o)
            m = evaluate(r, res)
            times.append(dt); ssims.append(m["ssim"]); psnrs.append(m["psnr"])
        chest_phantom_summary[name] = {
            "time": float(np.mean(times)), "ssim": float(np.mean(ssims)),
            "psnr": float(np.mean(psnrs)),
        }
        print(f"{name:<24}{np.mean(times)*1e3:>10.2f}{np.mean(ssims):>9.3f}"
              f"{np.mean(psnrs):>9.2f}")

    fig, axes = plt.subplots(1, 4, figsize=(11, 3))
    for ax, (k, t) in zip(axes, [("ref", "phantom GT"), ("obs", "degraded"),
                                 ("conventional", "conventional"), ("hybrid", "hybrid")]):
        im = {"ref": refs_p[0], "obs": obs_p[0],
              "conventional": method_conventional(obs_p[0])[0],
              "hybrid": method_hybrid(obs_p[0], qpar)[0]}[k]
        ax.imshow(im, cmap="gray", vmin=0, vmax=255); ax.set_title(t, fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()
    print("\nOn the anatomy-matched chest phantom the conventional pipeline")
    print("leads on SSIM -- the opposite of the clinical result above.")

    # ---------------------------------------------------------------
    # Mismatched HAND-phantom control (Sec. V-D of the paper): same
    # n, same protocol, wrong anatomy. This reproduces the ~0.5 SSIM
    # deficit number in code instead of only asserting it in prose.
    # ---------------------------------------------------------------
    rng_h = np.random.default_rng(43)
    refs_h = np.stack([hand_phantom(SIZE, rng_h) for _ in range(N_PHANTOM)])
    obs_h = np.stack([xray_degrade(r, rng_h) for r in refs_h])

    print(f"\nMismatched HAND phantom control (n={N_PHANTOM})")
    print(f"{'method':<24}{'SSIM':>9}{'PSNR':>9}")
    hand_phantom_summary = {}
    for name, fn in methods.items():
        ssims, psnrs = [], []
        for r, o in zip(refs_h, obs_h):
            res, _ = fn(o)
            m = evaluate(r, res)
            ssims.append(m["ssim"]); psnrs.append(m["psnr"])
        hand_phantom_summary[name] = {"ssim": float(np.mean(ssims)),
                                       "psnr": float(np.mean(psnrs))}
        print(f"{name:<24}{np.mean(ssims):>9.3f}{np.mean(psnrs):>9.2f}")

    deficit = (chest_phantom_summary["conventional"]["ssim"]
               - chest_phantom_summary["hybrid"]["ssim"])
    deficit_hand = (hand_phantom_summary["conventional"]["ssim"]
                     - hand_phantom_summary["hybrid"]["ssim"])
    print(f"\nConventional-vs-hybrid SSIM deficit: chest phantom {deficit:+.3f}, "
          f"hand phantom {deficit_hand:+.3f}")


## 10. Reproducibility notes

The original executed student's notebook recorded the clinical and phantom aggregate results copied into `../results/recorded_aggregate_metrics.csv`. This cleaned notebook intentionally contains **no saved cell outputs**, personal computer paths, or embedded images.

- The notebook uses Qiskit `Operator` and a NumPy matrix-vector multiplication; **not** a hardware backend or Aer per-image simulation.
- Direct classical QPAR and dense-unitary QPAR should agree up to 8-bit rounding; this is checked above on a deterministic example.
- Hardware-dependent wall-clock times can change across machines. Do not assume a locally rerun benchmark will reproduce the precise original timings.
- The source notebook did not contain code to reproduce the earlier mixing-strength Figure 3. The revised manuscript omits the unsupported curve.
- No clinical diagnostic study or DnCNN/U-Net baseline was performed.
